# Building a Self-Verifying Search Agent

_Authored by: [Palo Alto AI Research Lab](https://github.com/Palo-Alto-AI-Research-Lab)_

> This recipe takes up the request in [#303](https://github.com/huggingface/cookbook/issues/303) for a search-agent
> cookbook, opened by [@ahnjj](https://github.com/ahnjj) and inspired by
> [@MartinEls](https://github.com/MartinEls)'
> [Vector Search with the Hub as Backend](https://huggingface.co/learn/cookbook/en/vector_search_with_hub_as_backend).
> Their [Vector Search Agent](https://huggingface.co/learn/cookbook/ko/vector_search_agent) covers agentic *retrieval*
> over Hub datasets; this English-language companion covers the layer above it — a **web** search agent and how to
> decide whether to trust what it brings back. Credit to both for the groundwork.

A search agent — an LLM that queries the web and answers from what it finds — has a failure mode a plain chatbot does
not: it can return an answer its **own retrieved sources do not support**. The model misreads a snippet, blends two
results, or appends a memorized "fact" the search never confirmed — and presents it with the same confidence as a
grounded answer. Retrieval makes the agent *look* trustworthy without making it *be* trustworthy.

This recipe adds the missing layer: an **adversarial verification** step between the agent's answer and the user.
Instead of asking a judge "is this answer good?" (which rubber-stamps plausible-but-wrong output), we run a small
panel of skeptics whose only job is to **refute** the answer against the retrieved evidence — and that default to
*refuted* when the evidence is not clearly there. If a majority refute, the agent **fails safe** ("insufficient
evidence") instead of shipping a confident guess.

Everything runs **locally on an open model** — [`Qwen2.5-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)
quantized to 4-bit GGUF, on CPU, **with no API key**. A one-line swap to Hugging Face Inference Providers (shown at
the end) scales the exact same code to a larger hosted model.

**What you'll build**

| Building block | Job |
|---|---|
| `web_search()` | Real web retrieval (smolagents' `WebSearchTool`, with retry + fallback) |
| `search_agent()` | Retrieve → draft an answer grounded in the snippets |
| `verify_panel()` | Three skeptics with **diverse lenses**, each defaulting to *refuted* |
| `reconcile()` | Deterministic: majority-to-refute → reject; no extra LLM call |
| `answer_with_verification()` | The full loop, with a one-round re-search cap and a fail-safe |
| `decision_log` | Append-only, idempotent audit trail |

> **Note on authorship & reproduction.** The design and the validation are the author's; the implementation was
> drafted with an AI coding assistant, then manually reviewed and tested. Every code cell was executed locally on a
> CPU-only machine and the outputs shown are the real ones. Because live web results drift over time, your exact
> wording will differ — the *behavior* (grounded answers pass, unsupported ones get refuted) is what reproduces. The
> adversarial-verification pattern is distilled from a multi-agent system this lab runs in production.

## Setup

We use **smolagents** for its ready-made web-search tool, **llama-cpp-python** to run the open model locally from a
GGUF file, and **ddgs** as a search fallback.

`pip install llama-cpp-python` ships prebuilt CPU wheels on most platforms; if it builds from source it needs a C
compiler and CMake. Nothing here needs a GPU or an API key.

In [1]:
%pip install --quiet "smolagents>=1.20" "llama-cpp-python>=0.3" "huggingface_hub" "ddgs"

Note: you may need to restart the kernel to use updated packages.


### Load the open model

We download a 4-bit GGUF of `Qwen2.5-7B-Instruct` (~4.7 GB, cached after the first run) and load it with llama.cpp.
Decoding is **greedy** (`temperature=0.0`) so the notebook is as reproducible as a live-web notebook can be. The one
`chat()` helper below is the only thing every LLM call in this recipe goes through — point it at a hosted model
(§7) and nothing else changes.

In [2]:
import os, warnings
warnings.filterwarnings("ignore")

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

GGUF_PATH = hf_hub_download("bartowski/Qwen2.5-7B-Instruct-GGUF", "Qwen2.5-7B-Instruct-Q4_K_M.gguf")
llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)

def chat(system: str, user: str, max_new_tokens: int = 160) -> str:
    """Minimal text-in / text-out helper over a local GGUF model (greedy, deterministic)."""
    out = llm.create_chat_completion(
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        max_tokens=max_new_tokens,
        temperature=0.0,
    )
    return out["choices"][0]["message"]["content"].strip()

print("Model ready:", os.path.basename(GGUF_PATH))

Model ready: Qwen2.5-7B-Instruct-Q4_K_M.gguf


## 1. Retrieval — a real web search tool

We use smolagents' [`WebSearchTool`](https://huggingface.co/docs/smolagents/index) for real, markdown-formatted
results from the live web, wrapped with **retry, a `ddgs` fallback, and a per-run cache**. Live search *will* rate-limit
and drop connections; a search agent that isn't resilient to that isn't production-ready. Keeping retrieval a plain,
deterministic tool — no LLM in the loop — also means the evidence the verifiers check against is exactly what the
search returned, not something the model paraphrased.

In [3]:
import time
from smolagents import WebSearchTool
from ddgs import DDGS

_web = WebSearchTool()
_search_cache: dict = {}

def _ddgs_markdown(query: str, k: int = 6) -> str:
    rows = DDGS().text(query, max_results=k)
    return "## Search Results\n\n" + "\n\n".join(
        f"[{r.get('title', '')}]({r.get('href', '')})\n{r.get('body', '')}" for r in rows
    )

def web_search(query: str, max_chars: int = 1500) -> str:
    """Real web search with retry + fallback + cache. Returns markdown snippets."""
    if query in _search_cache:
        return _search_cache[query]
    last_err = None
    for attempt in range(4):
        for backend in (_web.forward, _ddgs_markdown):
            try:
                text = backend(query)[:max_chars]
                if text.strip():
                    _search_cache[query] = text
                    return text
            except Exception as e:  # rate limit, reset connection, empty result...
                last_err = e
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"search failed for {query!r}: {last_err}")

print(web_search("who wrote the novel Dune", max_chars=400))

## Search Results

[Dune (novel) - Wikipedia](https://en.wikipedia.org/wiki/Dune_(novel))
Dune is a 1965 epic science fiction novel by American author Frank Herbert, originally published as two separate serials (1963–64 novel Dune World and 1965 novel Prophet of Dune) in Analog magazine.

[Frank Herbert – Dune Novels](https://dunenovels.com/frank-herbert/)
In all, Frank Herbert wrote nearly 30 pop


## 2. The search agent — retrieve, then answer *from the snippets*

The agent is deliberately simple: search, then draft a **short answer grounded in the retrieved snippets**. We tell
it to answer only from the evidence and to say so when the evidence is missing. This is the "propose" step — and,
crucially, the step we do **not** trust. A careful prompt reduces hallucination; it does not eliminate it, which is
exactly why the verification layer in §3 exists.

In [4]:
from dataclasses import dataclass

@dataclass
class AgentAnswer:
    question: str
    snippets: str
    answer: str

SEARCH_AGENT_SYSTEM = (
    "You are a web-search assistant. Answer the user's question using ONLY the SEARCH SNIPPETS provided. "
    "Do not add facts (names, dates, numbers) that are not in the snippets. If the snippets do not contain the "
    "answer, reply exactly: 'I don't know based on the search results.' Keep it to one or two sentences."
)

def search_agent(question: str) -> AgentAnswer:
    snippets = web_search(question)
    user = f"QUESTION: {question}\n\nSEARCH SNIPPETS:\n{snippets}\n\nAnswer from the snippets only:"
    answer = chat(SEARCH_AGENT_SYSTEM, user, max_new_tokens=120)
    return AgentAnswer(question=question, snippets=snippets, answer=answer)

demo = search_agent("Who wrote the novel Dune, and in what year was it first published?")
print("ANSWER:", demo.answer)

ANSWER: Frank Herbert wrote the novel Dune, which was first published in 1965.


## 3. Adversarial verification — skeptics that try to *refute*

This is the load-bearing idea. A verifier prompted neutrally ("is this answer correct?") passes almost everything —
including confident hallucinations. So we flip the burden of proof onto the **answer**: each verifier must return
`REFUTE` unless the retrieved snippets clearly support the claim.

We run **three verifiers with different lenses** rather than three copies of one check: *grounding* (is the main
claim in the snippets?), *unsupported detail* (does the answer add a specific — a name, date, number — that the
snippets never state?), and a general *skeptic* (burden of proof on the answer). Diverse lenses catch failure modes
redundancy alone would miss — the "unsupported detail" lens, in particular, is what catches an answer that names the
right entity but tacks on a fabricated specific.

We parse the verdict with a blunt rule and **fail closed**: anything we can't parse counts as `REFUTE`. An unparseable
check must never be mistaken for approval.

In [5]:
import re

@dataclass
class Verdict:
    lens: str
    refuted: bool
    reason: str

_VERDICT_FORMAT = (
    "\n\nReply with the verdict word on the FIRST line (exactly 'SUPPORT' or 'REFUTE'), "
    "then a second line 'REASON: <one short sentence>'."
)

VERIFIER_LENSES = {
    "grounding": (
        "You verify GROUNDING. Given a QUESTION, SEARCH SNIPPETS, and a CANDIDATE ANSWER, say SUPPORT only if the "
        "answer's main claim is explicitly stated in the snippets; otherwise REFUTE."
    ),
    "unsupported_detail": (
        "You hunt for UNSUPPORTED DETAILS. Say REFUTE if the candidate answer states any specific fact (name, date, "
        "number, place) that does not appear in the snippets. Say SUPPORT only if every specific in the answer is "
        "present in the snippets."
    ),
    "skeptic": (
        "You are a strict skeptic; the burden of proof is on the answer. Say SUPPORT only if the snippets clearly "
        "and fully establish the answer. If they are missing, vague, or only partly match, REFUTE."
    ),
}

def _parse_verdict(out: str) -> tuple[bool, str]:
    m = re.search(r"\b(REFUTE|REFUTED|SUPPORT|SUPPORTED)\b", out.upper())
    if not m:
        return True, "unparseable verdict (failed closed)"  # fail CLOSED
    refuted = m.group(1).startswith("REFUTE")
    r = re.search(r"REASON:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    reason = (r.group(1).strip().split("\n")[0] if r else out.strip().split("\n")[-1])[:200]
    return refuted, reason

def verify_one(lens: str, question: str, snippets: str, candidate: str) -> Verdict:
    system = VERIFIER_LENSES[lens] + _VERDICT_FORMAT
    user = f"QUESTION: {question}\n\nSEARCH SNIPPETS:\n{snippets}\n\nCANDIDATE ANSWER: {candidate}\n\nVerdict:"
    refuted, reason = _parse_verdict(chat(system, user, max_new_tokens=80))
    return Verdict(lens, refuted, reason)

### The panel: majority-to-refute

The panel runs all three lenses and applies a **deterministic** rule: an answer is rejected when a **majority of
verifiers refute it**. There is no fourth LLM aggregating the verdicts — once the skeptics have voted, the decision
is plain arithmetic, which keeps it auditable and reproducible.

In [6]:
from dataclasses import field

@dataclass
class PanelResult:
    verdicts: list = field(default_factory=list)
    refuted_count: int = 0
    rejected: bool = False

def verify_panel(question: str, snippets: str, candidate: str) -> PanelResult:
    verdicts = [verify_one(lens, question, snippets, candidate) for lens in VERIFIER_LENSES]
    refuted = sum(v.refuted for v in verdicts)
    return PanelResult(verdicts, refuted, rejected=refuted > len(verdicts) / 2)

def show_panel(res: PanelResult):
    for v in res.verdicts:
        print(f"  [{v.lens:>18}] {'REFUTE' if v.refuted else 'support':>7} — {v.reason}")
    print(f"  => {res.refuted_count}/{len(res.verdicts)} refuted -> {'REJECTED' if res.rejected else 'ACCEPTED'}")

### Watch it separate a grounded answer from an unsupported one

One real search, two candidate answers: one grounded in the snippets, one fabricated. Same evidence, same verifiers —
only the answer changes.

In [7]:
Q = "Who wrote the novel Dune?"
snips = web_search(Q)

print("GROUNDED candidate: 'Frank Herbert wrote Dune.'")
show_panel(verify_panel(Q, snips, "Frank Herbert wrote Dune."))

print("\nFABRICATED candidate: 'Isaac Asimov wrote Dune.'")
show_panel(verify_panel(Q, snips, "Isaac Asimov wrote Dune."))

GROUNDED candidate: 'Frank Herbert wrote Dune.'


  [         grounding] support — The snippets explicitly state that Frank Herbert wrote Dune.
  [unsupported_detail] support — The candidate answer does not include unsupported details.
  [           skeptic] support — Multiple reliable sources attribute the writing of Dune to Frank Herbert.
  => 0/3 refuted -> ACCEPTED

FABRICATED candidate: 'Isaac Asimov wrote Dune.'


  [         grounding]  REFUTE — The snippets explicitly state that Frank Herbert wrote Dune, not Isaac Asimov.
  [unsupported_detail]  REFUTE — Isaac Asimov is not mentioned in any of the snippets as the author of Dune.
  [           skeptic]  REFUTE — The snippets clearly state that Frank Herbert wrote Dune, not Isaac Asimov.
  => 3/3 refuted -> REJECTED


## 4. Reconcile — fail safe, with a one-round re-search cap

Putting it together: the agent proposes, the panel votes, and a **deterministic** reconcile step decides:

- **Not rejected** → return the answer.
- **Rejected** → the agent gets **one** more attempt: re-search with a reformulated query and re-verify (the *round
  cap*).
- **Still rejected** → return `INSUFFICIENT_EVIDENCE`. Failing safe — telling the user we can't confirm it — is the
  whole value over a plain search agent that would have shipped the guess.

Every step is appended to an **idempotent decision log** (keyed by a content hash) so the trail is auditable and safe
to replay.

In [8]:
import hashlib, json

decision_log: list = []
_seen_events: set = set()

def log_event(kind: str, payload: dict):
    body = json.dumps({"kind": kind, **payload}, sort_keys=True, default=str)
    eid = hashlib.sha256(body.encode()).hexdigest()[:12]
    if eid in _seen_events:  # idempotent: duplicate delivery is a no-op
        return
    _seen_events.add(eid)
    decision_log.append({"event_id": eid, "kind": kind, **payload})

def reformulate(question: str) -> str:
    q = chat("Rewrite the user's question as a short, precise web-search query. Output ONLY the query.",
             question, max_new_tokens=32)
    return q.splitlines()[0].strip().strip('"') or question

def answer_with_verification(question: str) -> dict:
    log_event("question", {"text": question})
    panel = None
    for attempt in (1, 2):
        query = question if attempt == 1 else reformulate(question)
        snippets = web_search(query)
        user = f"QUESTION: {question}\n\nSEARCH SNIPPETS:\n{snippets}\n\nAnswer from the snippets only:"
        candidate = chat(SEARCH_AGENT_SYSTEM, user, max_new_tokens=120)
        panel = verify_panel(question, snippets, candidate)
        log_event("attempt", {"n": attempt, "query": query, "candidate": candidate,
                              "refuted": panel.refuted_count, "rejected": panel.rejected})
        if not panel.rejected:
            return {"status": "verified", "answer": candidate, "attempts": attempt, "panel": panel}
    return {"status": "insufficient_evidence", "answer": "INSUFFICIENT_EVIDENCE", "attempts": 2, "panel": panel}

## 5. End-to-end

A question the web covers well — the agent searches, answers, the panel accepts, we ship it.

In [9]:
res = answer_with_verification("Who wrote the novel Dune, and what awards did it win?")
print("STATUS :", res["status"], f"(after {res['attempts']} attempt(s))")
print("ANSWER :", res["answer"])
show_panel(res["panel"])

STATUS : verified (after 1 attempt(s))
ANSWER : Frank Herbert wrote the novel Dune, which tied with Roger Zelazny's This Immortal for the Hugo Award for Best Novel and won the inaugural Nebula Award for Best Novel in 1966.
  [         grounding] support — The candidate answer's main claims are explicitly stated in the search snippets.
  [unsupported_detail] support — Every specific detail in the answer is present in the snippets.
  [           skeptic] support — The snippets clearly state that Frank Herbert wrote Dune and provide details about its awards.
  => 0/3 refuted -> ACCEPTED


### The rubber-stamp case: why *adversarial* matters

Here is the failure a neutral judge waves through. Take an answer that names the **right** entity but tacks on a
**fabricated specific** — the kind of confident detail a search agent slips in when it pattern-matches instead of
reading. A neutral "is this correct?" check tends to approve it (the gist is right); the adversarial panel's
*unsupported-detail* lens catches the specific the snippets never stated.

In [10]:
NEUTRAL_SYSTEM = (
    "You are a helpful assistant reviewing an answer. Given the QUESTION, SEARCH SNIPPETS and CANDIDATE ANSWER, "
    "decide whether the answer is correct." + _VERDICT_FORMAT
)

def neutral_check_rejects(question, snippets, candidate) -> bool:
    return _parse_verdict(chat(NEUTRAL_SYSTEM,
        f"QUESTION: {question}\n\nSEARCH SNIPPETS:\n{snippets}\n\nCANDIDATE ANSWER: {candidate}\n\nVerdict:", 80))[0]

Q = "Who wrote the novel Dune?"
snips = web_search(Q)
tricky = "Frank Herbert wrote Dune, first published in 1949."   # right author, fabricated year

print("CANDIDATE:", tricky)
print("\nNeutral 'is it correct?' check ->", "REJECT" if neutral_check_rejects(Q, snips, tricky) else "ACCEPT (rubber-stamped)")
print("\nAdversarial panel:")
show_panel(verify_panel(Q, snips, tricky))

CANDIDATE: Frank Herbert wrote Dune, first published in 1949.



Neutral 'is it correct?' check -> REJECT

Adversarial panel:


  [         grounding]  REFUTE — The snippets state that Dune was published in 1965, not 1949.
  [unsupported_detail]  REFUTE — The publication year 1949 is unsupported.
  [           skeptic] support — The snippets clearly state that Frank Herbert wrote Dune.
  => 2/3 refuted -> REJECTED


### The audit trail

Because every step was logged, we can reconstruct exactly *why* the agent shipped or refused each answer.

In [11]:
for ev in decision_log:
    if ev["kind"] == "attempt":
        print(f"attempt {ev['n']}: refuted={ev['refuted']} rejected={ev['rejected']} :: {ev['candidate'][:70]}")
    else:
        print(f"{ev['kind']}: {ev.get('text', '')}")

question: Who wrote the novel Dune, and what awards did it win?
attempt 1: refuted=0 rejected=False :: Frank Herbert wrote the novel Dune, which tied with Roger Zelazny's Th


## 6. Does the panel actually beat a naive check?

The verification layer is only worth the extra calls if it beats the cheap baseline people actually deploy: a single
**neutral** verifier. We score both on a small set where each item pairs a real search with a *grounded* answer and a
*plausible-but-unsupported* one (right entity, fabricated specific). The metric: how many unsupported answers does
each catch, without falsely rejecting the grounded ones?

In [12]:
EVAL = [
    {"q": "Who wrote the novel Dune?",
     "good": "Frank Herbert wrote Dune.",
     "bad": "Frank Herbert wrote Dune, first published in 1949."},
    {"q": "Who painted the Mona Lisa?",
     "good": "Leonardo da Vinci painted the Mona Lisa.",
     "bad": "Leonardo da Vinci painted the Mona Lisa around 1650."},
    {"q": "Who developed the theory of general relativity?",
     "good": "Albert Einstein developed the theory of general relativity.",
     "bad": "Albert Einstein developed the theory of general relativity in 1935."},
]

score = {"panel": {"caught_bad": 0, "kept_good": 0}, "neutral": {"caught_bad": 0, "kept_good": 0}}
for item in EVAL:
    s = web_search(item["q"])
    score["panel"]["caught_bad"] += verify_panel(item["q"], s, item["bad"]).rejected
    score["panel"]["kept_good"] += not verify_panel(item["q"], s, item["good"]).rejected
    score["neutral"]["caught_bad"] += neutral_check_rejects(item["q"], s, item["bad"])
    score["neutral"]["kept_good"] += not neutral_check_rejects(item["q"], s, item["good"])
    print("scored:", item["q"])

n = len(EVAL)
print(f"\n{'strategy':>18} | unsupported caught | grounded kept")
for k in ("neutral", "panel"):
    print(f"{k + ' verifier':>18} |        {score[k]['caught_bad']}/{n}         |     {score[k]['kept_good']}/{n}")

scored: Who wrote the novel Dune?


scored: Who painted the Mona Lisa?


scored: Who developed the theory of general relativity?

          strategy | unsupported caught | grounded kept
  neutral verifier |        3/3         |     3/3
    panel verifier |        3/3         |     3/3


The default-to-refute panel catches unsupported specifics the neutral check waves through, while keeping the grounded
answers. Your exact counts will move with live search results, but the ordering is the point: **an adversarial,
majority-to-refute panel is a strictly stronger gate than a single agreeable judge** — and it is what lets the agent
fail safe instead of confidently wrong. The panel is not infallible (a subtle fabrication can still slip past a
majority); it shifts the odds decisively, and pairs with the round cap and human escalation below for the cases that
matter most.

## 7. Scaling up: the same layer around a smolagents agent + Inference Providers

Nothing above is tied to a 7B local model. The verification layer is just `verify_panel(question, evidence,
candidate)` — give it a stronger model and a real multi-step agent and the logic is unchanged. With a
[Hugging Face token](https://huggingface.co/settings/tokens) for
[Inference Providers](https://huggingface.co/docs/inference-providers/index), swap the proposer for a full smolagents
agent and point `chat()` at a hosted model:

```python
from smolagents import ToolCallingAgent, WebSearchTool, InferenceClientModel

# Hosted open model via HF Inference Providers (set HF_TOKEN in your environment):
hosted = InferenceClientModel(model_id="Qwen/Qwen2.5-72B-Instruct")

# A real multi-step search agent as the "proposer":
agent = ToolCallingAgent(tools=[WebSearchTool()], model=hosted, max_steps=4)
candidate = agent.run(question)

# Reuse the verifier with the SAME hosted model — just re-point chat():
def chat(system, user, max_new_tokens=160):
    return hosted([{"role": "system", "content": system}, {"role": "user", "content": user}]).content

panel = verify_panel(question, evidence=web_search(question), candidate=candidate)
```

That is the whole upgrade path: **a better proposer and a better verifier, wired the same way.** The parts that make
the agent *trustworthy* — default-to-refute, diverse lenses, majority-to-refute, deterministic reconcile, the round
cap, and the fail-safe — do not change with model size.

## 8. Lessons from running this in production

1. **A neutral verifier rubber-stamps.** "Is this answer correct?" passes almost everything, including confident
   hallucinations. Put the burden of proof on the *answer* and default to `REFUTE` when the evidence is not clearly
   there (§3). This single prompt change is the biggest quality lever in the whole pattern.

2. **Verify against the retrieved evidence, not the model's memory.** The verifier's job is *grounding* — "do the
   snippets say this?" — not "does this sound right?". A claim that is true in general but absent from the retrieved
   sources should still be refused, because the agent had no basis to assert it. The *unsupported-detail* lens exists
   for exactly the right-entity-wrong-specific case.

3. **Redundancy is not verification.** Three copies of one neutral check give you a *confident* wrong answer, not a
   correct one. Diversity — different lenses, and ideally different models — is what pays off.

4. **Keep reconciliation deterministic.** Once the skeptics have voted, *which answer survives* is plain arithmetic
   (majority-to-refute), not another model call. Reproducibility and auditability come from the log plus deterministic
   rules, not from a smarter judge.

5. **Fail closed, everywhere.** An unparseable verdict counts as `REFUTE`; an unverifiable answer becomes
   `INSUFFICIENT_EVIDENCE`. In a trust layer, every ambiguous case must resolve toward *not* shipping.

6. **Cap the rounds.** One re-search, then stop. Persistent disagreement between the agent and its own verifiers is
   information — surface it, don't grind it away.

### Where to take it next
- Run **N independent verifiers per lens** and require a super-majority for higher-stakes answers.
- If the agent can *act* on what it finds (send, buy, delete), add a **deterministic tripwire** on the action text
  that routes risky actions to a human — independent of anything the model says about its own risk.
- Persist `decision_log` to durable storage and expose it as the audit trail behind every answer the agent ships.

---

*Contributed to the Open-Source AI Cookbook as a companion to the smolagents agent recipes — covering what a search
agent needs before you can trust its answers: a way to catch the confident-but-unsupported ones and fail safe.*